# NDVI and land cover change in Tinigua National Natural Park (Colombia)

**Question:** how much did vegetation (Sentinel-2 NDVI) drop inside Tinigua National Natural Park between the dry seasons of 2019 and 2025, on which land cover, and how did it evolve year by year?

**Rule of this project:** the CRS of every layer is checked before any area, distance or buffer calculation, and every result states the CRS used and how many pixels had valid data.

**Tooling:** Sentinel-2 bands are read in place from public COGs with [terrio](https://github.com/opengeos/terrio) (pure Rust, no GDAL). Because terrio is young (v0.x), section 4 cross-checks it against GDAL on real data before using it.

## 0. Setup

Works both locally and in Google Colab. In Colab the repository is cloned so that `src/` can be imported, and the libraries that Colab does not preinstall are added.

In [ ]:
import sys, os, subprocess

REPO = "ndvi-sentinel2-redd"  # replace with the repository name
IN_COLAB = "google.colab" in sys.modules

if IN_COLAB:
    # Colab only opens the notebook: clone the repo once and work from its root.
    # Safe to run again or after "Restart session".
    if os.path.basename(os.getcwd()) != REPO:
        if not os.path.exists(REPO):
            subprocess.run(["git", "clone", f"https://github.com/albertolozada/{REPO}.git"], check=True)
        os.chdir(REPO)
    # Libraries not preinstalled in Colab
    subprocess.run([sys.executable, "-m", "pip", "install", "-q",
                    "terrio[xarray]", "pystac-client", "rioxarray",
                    "contextily", "matplotlib-scalebar"], check=True)
elif os.path.basename(os.getcwd()) == "notebooks":
    os.chdir("..")

sys.path.insert(0, os.getcwd())

import numpy as np
import pandas as pd
import geopandas as gpd
import matplotlib.pyplot as plt
import terrio
from terrio.rio.enums import Resampling

from src.geo_checks import check_crs, to_metric, area_ha
from src.maps import plot_aoi_map
from src import s2, sources
from src.validation import compare_window

print("terrio", terrio.__version__)

### Parameters

Change these to reuse the notebook for another protected area or period. Seasons are named by the year they end: the 2019 dry season runs from December 2018 to March 2019.

In [ ]:
AREA_NAME = "Tinigua"                     # part of the name in RUNAP
AREA_CATEGORY = "Parque Nacional Natural"

BEFORE, AFTER = 2019, 2025                # dry seasons compared
SERIES_YEARS = list(range(2019, 2026))    # dry seasons in the time series
DRY_MONTHS = [12, 1, 2, 3]                # dry season in the north-west Amazon (Meta)

MAX_CLOUD = 60      # % scene cloud cover in the STAC search (clouds are masked per pixel later)
RES_MAIN = 40       # m, grid of the before/after comparison
RES_SERIES = 100    # m, grid of the time series (coarser = faster)

NDVI_FOREST = 0.6   # NDVI before, above which a pixel is treated as dense vegetation
LOSS_DROP = -0.2    # NDVI change below which a dense-vegetation pixel is flagged as loss

os.makedirs("data", exist_ok=True)
os.makedirs("outputs/maps", exist_ok=True)

def season(year):
    """STAC datetime range of the dry season that ends in `year`."""
    return f"{year - 1}-12-01/{year}-03-31"

## 1. Load the study area

The official boundary comes from RUNAP (Registro Único Nacional de Áreas Protegidas, Parques Nacionales Naturales de Colombia). It is downloaded once and cached in `data/`, which is not versioned. If the service is down, save the boundary by hand as `data/aoi.geojson` and run the cell again.

In [ ]:
cache = "data/aoi.geojson"
aoi = sources.load_runap(AREA_NAME, AREA_CATEGORY, cache=cache)
print(len(aoi), "feature(s)")
aoi.drop(columns="geometry")

## 2. Check the CRS and the area

RUNAP serves the layer in geographic coordinates (degrees), so it is reprojected to its UTM zone before measuring. The calculated area is compared with the area in the declaration resolution.

In [ ]:
check_crs(aoi, "aoi (RUNAP)")

In [ ]:
CRS = aoi.estimate_utm_crs().to_string()
aoi_m = to_metric(aoi, CRS)
print("CRS used for measurements and for the analysis grid:", CRS)

area_calc = float(area_ha(aoi_m).sum())
area_decl = float(aoi["area_ha_total_resolucion"].sum())
print(f"Calculated area ({CRS}): {area_calc:,.1f} ha")
print(f"Area in the RUNAP resolution field: {area_decl:,.1f} ha")
print(f"Difference: {area_calc - area_decl:,.1f} ha ({100 * (area_calc - area_decl) / area_decl:.2f} %)")
check_crs(aoi_m, "aoi (metric)")

### Location map

Study area over a satellite basemap, with scale bar, north arrow and basemap attribution.

In [ ]:
plot_aoi_map(aoi, out_png="outputs/maps/location_map.png", basemap="satellite",
             title="Tinigua National Natural Park (RUNAP boundary)", dpi=120)
plt.show()

## 3. Find Sentinel-2 scenes

Sentinel-2 L2A (surface reflectance) from the [Earth Search](https://earth-search.aws.element84.com/v1) STAC API, dry season only. The table lists every scene used, so the result can be traced back to its inputs.

In [ ]:
bbox = tuple(aoi.total_bounds)  # EPSG:4326
items_before = s2.search_items(bbox, season(BEFORE), MAX_CLOUD, DRY_MONTHS)
items_after = s2.search_items(bbox, season(AFTER), MAX_CLOUD, DRY_MONTHS)
print(f"{BEFORE}: {len(items_before)} scenes | {AFTER}: {len(items_after)} scenes")
s2.items_table(items_before + items_after)

## 4. Cross-check terrio against GDAL

The same 5 x 5 km window of one red band is read with terrio and with rioxarray (GDAL), at native resolution and without resampling, from the least cloudy scene that covers the centre of the park. Both should return identical pixels.

In [ ]:
import terrio.xarray as txr
from pyproj import Transformer
from shapely.geometry import Point, shape

# Centre of the park, in EPSG:4326, to pick a scene that actually covers it
cx, cy = aoi_m.geometry.union_all().centroid.coords[0]
lon, lat = Transformer.from_crs(CRS, "EPSG:4326", always_xy=True).transform(cx, cy)

# Each Sentinel-2 item is one 100 x 100 km MGRS tile: use the least cloudy one that contains the centre
covering = [i for i in items_after if shape(i.geometry).contains(Point(lon, lat))]
if not covering:
    raise ValueError("No scene of the after season covers the centre of the park")
item = min(covering, key=lambda i: i.properties.get("eo:cloud_cover", 100))

href = item.assets["red"].href
item_crs = txr.open_rasterio(href).terrio.crs.to_string()  # CRS of the scene itself

# a 5 x 5 km window centred on the park, in the CRS of the scene
x, y = Transformer.from_crs("EPSG:4326", item_crs, always_xy=True).transform(lon, lat)
check = compare_window(href, (x - 2500, y - 2500, x + 2500, y + 2500), item_crs)
print(item.id, "| cloud cover", item.properties.get("eo:cloud_cover"), "%")
check

## 5. Dry-season NDVI composites (before and after)

For each scene, red (B04) and NIR (B08) are read from the COG overview closest to the grid size, converted to reflectance, masked with the Scene Classification Layer and resampled onto one grid. The composite is the per-pixel median of the clear observations.

Products processed with baseline 04.00 or later (from 25 January 2022) can carry a +1000 DN offset, and the STAC metadata does not always state it consistently with the stored values. The offset is therefore decided per scene from the pixels: it is subtracted when the 5th percentile of clear red DN is 1000 or more. The log below records the declared and the applied offset of every scene.

In [ ]:
grid = s2.make_grid(aoi, CRS, RES_MAIN)
print(f"Grid: {grid.shape[1]} x {grid.shape[0]} pixels of {grid.res} m, {grid.crs}")

before, log_before = s2.composite(items_before, grid, aoi)
after, log_after = s2.composite(items_after, grid, aoi)

summary = pd.DataFrame([s2.ndvi_summary(before, grid, str(BEFORE)),
                        s2.ndvi_summary(after, grid, str(AFTER))])
summary

In [ ]:
pd.concat([log_before, log_after])  # per scene: declared and applied offset, clear pixels

**Consistency check.** Dense vegetation that did not change should keep almost the same NDVI in both seasons.

In [ ]:
stable = (before.ndvi > 0.7) & (after.ndvi > 0.7)
print(f"Stable dense vegetation: {float(stable.sum()) * grid.pixel_area_ha:,.0f} ha")
print(f"Median NDVI {BEFORE}: {float(before.ndvi.where(stable).median()):.3f} | "
      f"{AFTER}: {float(after.ndvi.where(stable).median()):.3f}")

In [ ]:
def show(ax, da, title, **kw):
    b = grid.bounds
    im = ax.imshow(da.values, extent=(b[0], b[2], b[1], b[3]), interpolation="nearest", **kw)
    aoi_m.boundary.plot(ax=ax, color="black", linewidth=0.6)
    ax.set_title(title, fontsize=10)
    ax.set_axis_off()
    return im

fig, axes = plt.subplots(1, 3, figsize=(16, 6), layout="constrained")
show(axes[0], before.ndvi, f"NDVI dry season {BEFORE}", cmap="RdYlGn", vmin=0, vmax=1)
im = show(axes[1], after.ndvi, f"NDVI dry season {AFTER}", cmap="RdYlGn", vmin=0, vmax=1)
fig.colorbar(im, ax=axes[1], shrink=0.7, label="NDVI")
im2 = show(axes[2], after.n_clear, f"Clear observations {AFTER}", cmap="viridis")
fig.colorbar(im2, ax=axes[2], shrink=0.7, label="scenes")
fig.text(0.99, 0.01, f"CRS {CRS}, {grid.res} m. Sentinel-2 L2A (Copernicus), read with terrio.",
         ha="right", fontsize=7, color="0.3")
plt.savefig("outputs/maps/ndvi_before_after.png", dpi=120, bbox_inches="tight")
plt.show()

## 6. NDVI change and potential vegetation loss

A pixel is flagged as **potential vegetation loss** when its NDVI was above `NDVI_FOREST` in the first season and dropped by more than `LOSS_DROP`. This is a signal to look at, not a deforestation map: NDVI also changes with phenology, drought, residual clouds or burnt but regrowing areas.

In [ ]:
dndvi = after.ndvi - before.ndvi
loss = (before.ndvi > NDVI_FOREST) & (dndvi < LOSS_DROP)
both = before.ndvi.notnull() & after.ndvi.notnull()

loss_ha = float(loss.sum()) * grid.pixel_area_ha
compared_ha = float(both.sum()) * grid.pixel_area_ha
print(f"Pixels with valid NDVI in both seasons: {compared_ha:,.0f} ha "
      f"({100 * compared_ha / area_calc:.1f} % of the calculated area)")
print(f"Potential vegetation loss {BEFORE}-{AFTER}: {loss_ha:,.0f} ha "
      f"(grid {grid.res} m, {CRS})")

In [ ]:
fig, ax = plt.subplots(figsize=(8, 8))
im = show(ax, dndvi, f"NDVI change, dry season {BEFORE} to {AFTER}", cmap="RdBu", vmin=-0.5, vmax=0.5)
fig.colorbar(im, ax=ax, shrink=0.7, label="NDVI change")
fig.text(0.99, 0.01, f"CRS {CRS}, {grid.res} m. Sentinel-2 L2A (Copernicus), read with terrio.",
         ha="right", fontsize=7, color="0.3")
plt.savefig("outputs/maps/ndvi_change.png", dpi=120, bbox_inches="tight")
plt.savefig("outputs/maps/preview.png", dpi=100, bbox_inches="tight")  # image under the README title
plt.show()

## 7. Land cover of the change (ESA WorldCover 2021)

ESA WorldCover 10 m 2021 is resampled onto the same grid with the majority (mode) rule and crossed with the loss mask. It is read at full resolution, because averaged overviews would mix class codes.

In [ ]:
wc_urls = sources.worldcover_urls(aoi.total_bounds)
print("\n".join(wc_urls))
wc = None
for url in wc_urls:
    part = s2.read_to_grid(url, grid, Resampling.mode)
    if part is not None:
        wc = part if wc is None else wc.fillna(part)
wc = s2.mask_to_aoi(wc, aoi, grid)

table = pd.DataFrame({
    "class": wc.values[np.isfinite(wc.values)].astype(int),
    "loss": loss.values[np.isfinite(wc.values)],
})
by_class = table.groupby("class").agg(area_ha=("loss", "size"), loss_ha=("loss", "sum"))
by_class *= grid.pixel_area_ha
by_class["loss_%_of_class"] = (100 * by_class["loss_ha"] / by_class["area_ha"]).round(2)
by_class.insert(0, "name", sources.class_names(by_class.index))
by_class.round(1)

## 8. Year-by-year series (dry seasons)

The same composite on a coarser grid (`RES_SERIES`) for every dry season in `SERIES_YEARS`. This is the slowest step: each season reads every scene, so expect several minutes per season in Colab. Shorten `SERIES_YEARS` for a quick test.

In [ ]:
grid_s = s2.make_grid(aoi, CRS, RES_SERIES)
rows = []
for year in SERIES_YEARS:
    items = s2.search_items(bbox, season(year), MAX_CLOUD, DRY_MONTHS)
    ds, _ = s2.composite(items, grid_s, aoi, verbose=False)
    row = s2.ndvi_summary(ds, grid_s, str(year))
    row["scenes"] = len(items)
    rows.append(row)
    print(row)

series = pd.DataFrame(rows).set_index("period")
series

In [ ]:
fig, ax1 = plt.subplots(figsize=(8, 4))
ax1.plot(series.index, series["median_ndvi"], marker="o", color="#2f6b4f")
ax1.set_ylabel("Median NDVI (dry season)")
ax2 = ax1.twinx()
ax2.bar(series.index, series["ndvi_lt_0.5_ha"], alpha=0.25, color="#c1121f")
ax2.set_ylabel("Area with NDVI < 0.5 (ha)")
ax1.set_title(f"Tinigua NNP, dry-season NDVI ({RES_SERIES} m grid, {CRS})")
plt.savefig("outputs/maps/ndvi_series.png", dpi=120, bbox_inches="tight")
plt.show()

## 9. Results and limitations

*Fill in this section by hand after running the analysis: the code computes the numbers, this section explains what they mean. Use the CRS and the counts printed above.*

- **Main result:**
- **CRS used:**
- **Completeness:** share of the area with valid NDVI in both seasons (section 6) and scenes that failed (logs in section 5)
- **terrio vs GDAL check:** (section 4)
- **Limitations:**
  - NDVI loss is not deforestation: compare with official data (IDEAM, SMByC) or Global Forest Watch before drawing conclusions.
  - Persistent cloud in the Amazon leaves pixels with few or no clear observations.
  - ESA WorldCover is a single date (2021), between the two seasons compared.
  - Results depend on the grid size (`RES_MAIN`) and on the thresholds `NDVI_FOREST` and `LOSS_DROP`.